# Vision Privacy & Identity Lab – Module 2
## Training Setup

This notebook:
1. Installs training dependencies (diffusers, accelerate, xformers, bitsandbytes)
2. Generates BLIP captions with a unique activation token
3. Configures VRAM-saving options: **Gradient Checkpointing** + **xformers**
4. Launches a DreamBooth / LoRA training run

In [ ]:
# ── 1. Install training stack ─────────────────────────────────────────────────
!pip install -q \
    torch torchvision \
    transformers accelerate diffusers \
    xformers bitsandbytes \
    Pillow tqdm

In [ ]:
# ── 2. Colab + path setup ─────────────────────────────────────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/IA-'):
        !git clone -q https://github.com/ap-xlr8/IA- /content/IA-
    sys.path.insert(0, '/content/IA-')
else:
    sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

In [ ]:
# ── 3. Configuration ──────────────────────────────────────────────────────────
PROCESSED_DIR  = '/content/drive/MyDrive/vp_lab/processed' if IN_COLAB else './data/processed'
CAPTIONS_DIR   = '/content/drive/MyDrive/vp_lab/captions'  if IN_COLAB else './data/captions'
OUTPUT_MODEL   = '/content/drive/MyDrive/vp_lab/model'      if IN_COLAB else './model'

CLASE          = 'person'      # class word
DEVICE         = 'cuda'        # 'cuda' or 'cpu'
MAX_TOKENS     = 50

# Training hyper-parameters
BASE_MODEL     = 'runwayml/stable-diffusion-v1-5'
TRAIN_STEPS    = 800
LEARNING_RATE  = 1e-4
TRAIN_BATCH    = 1
GRAD_ACCUM     = 4

# VRAM savers
USE_GRADIENT_CHECKPOINTING = True
USE_XFORMERS               = True
USE_8BIT_ADAM              = True   # requires bitsandbytes
MIXED_PRECISION            = 'fp16' # 'no' | 'fp16' | 'bf16'

print('Configuration OK')

In [ ]:
# ── 4. Generate captions with BLIP + activation token ─────────────────────────
from caption_cleaner import generar_token_activacion, procesar_directorio

TOKEN = generar_token_activacion()
print(f'Activation token: {TOKEN}')

captions = procesar_directorio(
    image_dir=PROCESSED_DIR,
    output_dir=CAPTIONS_DIR,
    token=TOKEN,
    clase=CLASE,
    device=DEVICE,
    max_new_tokens=MAX_TOKENS,
)
print(f'\n✅  {len(captions)} captions generated in {CAPTIONS_DIR}')

In [ ]:
# ── 5. VRAM report ────────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / 1024**3
    print(f'GPU: {props.name}')
    print(f'VRAM: {total_gb:.1f} GB')
    if total_gb < 16:
        print('⚠️  < 16 GB VRAM detected – VRAM savers are strongly recommended.')
else:
    print('No CUDA GPU detected – training on CPU will be very slow.')

In [ ]:
# ── 6. Build accelerate / diffusers training command ─────────────────────────
# This cell assembles a `train_dreambooth_lora.py` invocation.
# The script is from: https://github.com/huggingface/diffusers/tree/main/examples/dreambooth

import shlex

cmd_parts = [
    'accelerate launch',
    '--mixed_precision', MIXED_PRECISION,
    'train_dreambooth_lora.py',
    '--pretrained_model_name_or_path', BASE_MODEL,
    '--instance_data_dir', PROCESSED_DIR,
    '--output_dir', OUTPUT_MODEL,
    '--instance_prompt', f'"{TOKEN} {CLASE}"',
    '--resolution', '512',
    '--train_batch_size', str(TRAIN_BATCH),
    '--gradient_accumulation_steps', str(GRAD_ACCUM),
    '--learning_rate', str(LEARNING_RATE),
    '--lr_scheduler', 'constant',
    '--lr_warmup_steps', '0',
    '--max_train_steps', str(TRAIN_STEPS),
]

if USE_GRADIENT_CHECKPOINTING:
    cmd_parts.append('--gradient_checkpointing')
if USE_XFORMERS:
    cmd_parts.append('--enable_xformers_memory_efficient_attention')
if USE_8BIT_ADAM:
    cmd_parts.append('--use_8bit_adam')

TRAIN_CMD = ' '.join(cmd_parts)
print('Training command:')
print(TRAIN_CMD)

In [ ]:
# ── 7. Download diffusers training script (if not present) ────────────────────
SCRIPT_URL = (
    'https://raw.githubusercontent.com/huggingface/diffusers/main/'
    'examples/dreambooth/train_dreambooth_lora.py'
)
if not os.path.exists('train_dreambooth_lora.py'):
    !wget -q {SCRIPT_URL}
    print('Script downloaded.')
else:
    print('Script already present.')

In [ ]:
# ── 8. Launch training ────────────────────────────────────────────────────────
# Uncomment the line below when you are ready to start training.
# Training can take 10–60 min depending on GPU and number of steps.

# !{TRAIN_CMD}
print('⚠️  Training command is commented out. Uncomment cell above to run.')